[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03a_baseline.ipynb)

# 03a — Baselines: Random Search and a Rule-Based Sweep

**Question.** What does the tilt space offer without a model?

- **Random search** evaluates Sobol points over the tilt box, with the same evaluation budget and the same
  initial design as TuRBO. Any gap between the two is what the model contributes.
- **Rule-based sweep** gives every cell on a band the same tilt and tunes the three band tilts by coordinate
  descent, the way an operator would. It is deterministic and not budget-matched.

**Outputs.** One run directory per method and seed under `outputs/optim/`. **Requirements.** A CUDA GPU.
Random search runs once per seed in `SEEDS` (`optim.seed`, the global `seed` in `configs/config.yaml`, unless `BAND_TILT_SEEDS` is set); the shell equivalent is `task baseline`.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna.rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

from functools import partial

import pandas as pd

from src.config import load_config
from src.evaluation import compare
from src.evaluation import runs as run_store
from src.evaluation.export import readable, save_table
from src.optim.run import run
from src.optim.space import TiltSpace
from src.utils.plotting import label, save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/03a_baseline"))
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/03a_baseline"))
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

# Search seeds: optim.seed from the config; BAND_TILT_SEEDS (space-separated) overrides it.
SEEDS = [int(seed) for seed in os.environ.get("BAND_TILT_SEEDS", str(cfg.optim.seed)).split()]

## 1. Search Space

In [3]:
space = TiltSpace.from_config(cfg)
search_space = (
    space.as_frame(space.baseline)
    .groupby("band", sort=False)
    .agg(
        cells=("cell", "size"),
        current=("tilt_deg", "median"),
        minimum=("tilt_min_deg", "min"),
        maximum=("tilt_max_deg", "max"),
    )
    .reset_index()
    .rename(
        columns={
            "cells": "Cells",
            "current": "Current tilt [°]",
            "minimum": "Minimum tilt [°]",
            "maximum": "Maximum tilt [°]",
        }
    )
)
search_space = readable(search_space)
save_table(search_space, "search_space")
print(f"{space.n_dim} decision variables: {len(space.cells)} cells x {len(space.band_names)} bands")
search_space

36 decision variables: 12 cells x 3 bands


,Band,Cells,Current tilt [°],Minimum tilt [°],Maximum tilt [°]
0,2600 MHz,12,10.0,0.0,10.0
1,1800 MHz,12,9.0,0.0,10.0
2,700 MHz,12,8.0,0.0,10.0


## 2. Runs

Each run evaluates the current configuration first, then searches, then archives its best configuration. A run already on disk for the same method and seed is reused; delete its directory under `outputs/optim/` to repeat it.

In [4]:
def on_disk():
    """The newest finished run of each method and seed."""
    return run_store.latest_per_method_and_seed(run_store.discover(cfg.optim.output.dir))


done = {(r.method, r.seed) for r in on_disk()}
for seed in SEEDS:
    if ("random", seed) not in done:
        run(load_config(overrides=["optim/method=random", f"optim.seed={seed}", *CONFIG_OVERRIDES]))

# Deterministic, so one run; its seed only labels it.
if not any(method == "rule" for method, _ in done):
    run(load_config(overrides=["optim/method=rule", f"optim.seed={SEEDS[0]}", *CONFIG_OVERRIDES]))

runs = [r for r in on_disk() if r.method == "rule" or (r.method == "random" and r.seed in SEEDS)]

jitc_llvm_init(): LLVM API initialization failed ..



random: 8 solutions published

 solution  recommended   score  hole_rate  overlap_rate  served_ratio  weak_rate
        0        False -0.5041     0.1618        0.3531        0.7758     0.3491
        1         True -0.3579     0.1549        0.3578        0.8208     0.3064
        2        False -0.3687     0.1536        0.3615        0.8188     0.3073
        3        False -0.3727     0.1543        0.3640        0.8187     0.3007
        4        False -0.3765     0.1553        0.3641        0.8213     0.3059
        5        False -0.3833     0.1534        0.3653        0.8151     0.3041
        6        False -0.3848     0.1557        0.3641        0.8202     0.3102
        7        False -0.3870     0.1546        0.3679        0.8231     0.3110

wrote outputs\optim\random\2026-09-15_14-28-43
  history: outputs\optim\random\2026-09-15_14-28-43\history.parquet
  best_tilt: outputs\optim\random\2026-09-15_14-28-43\best_tilt.parquet
  run: outputs\optim\random\2026-09-15_14-28-43\run


rule: 8 solutions published

 solution  recommended   score  hole_rate  overlap_rate  served_ratio  weak_rate
        0        False -0.5041     0.1618        0.3531        0.7758     0.3491
        1         True -0.3310     0.1516        0.3615        0.8279     0.2958
        2        False -0.3334     0.1511        0.3631        0.8272     0.2939
        3        False -0.3334     0.1511        0.3631        0.8272     0.2939
        4        False -0.3354     0.1516        0.3624        0.8271     0.2958
        5        False -0.3356     0.1514        0.3635        0.8280     0.2954
        6        False -0.3407     0.1517        0.3623        0.8245     0.2958
        7        False -0.3450     0.1526        0.3605        0.8221     0.2974

wrote outputs\optim\rule\2026-09-15_14-41-55
  history: outputs\optim\rule\2026-09-15_14-41-55\history.parquet
  best_tilt: outputs\optim\rule\2026-09-15_14-41-55\best_tilt.parquet
  run: outputs\optim\rule\2026-09-15_14-41-55\run.json

cho

## 3. Results per Run

The weighted score (`configs/kpi.yaml` `weights`) selects each run's best configuration.

In [5]:
incumbent = runs[0].incumbent_kpi

baseline_results = readable(compare.method_table(runs, cfg).drop(columns=["run"]))
save_table(baseline_results, "baseline_results")
print("Current configuration:", {label(k): round(v, 4) for k, v in incumbent.as_dict().items()})
baseline_results

Current configuration: {'Coverage hole rate': 0.1618, 'Co-band overlap rate': 0.3531, 'Served UE ratio': 0.7758, 'Weak coverage rate': 0.3491}


,Method,Seed,Evaluations,Best evaluation,Ray tracing [min],Wall clock [min],Coverage hole rate,Co-band overlap rate,Served UE ratio,Weak coverage rate,Weighted score,KPIs improved,KPIs worsened
0,Random search,42,145,125,11.1659,13.1871,0.1549,0.3578,0.8208,0.3064,-0.3579,3,1
1,Rule-based sweep,42,27,11,0.8256,1.0564,0.1516,0.3615,0.8279,0.2958,-0.3310,3,1


**Observations.** _To be written._